![Platform+](images/banner.png)

# LAB 6: MARKETPLACE
## Govern data you don't own — access control for a zero-copy share

> **Persona track: Platform Admin** — same lab, your angle.

:point_right: **In this lesson, you'll access a Private Listing shared with your account, then inspect the access posture — who can read acquired shared data, and how provider and consumer split control.**

The scenario: Roastery & Co. is opening a reading corner in their cafés. A fictional publisher — **Alpine Peaks Publishing** — has shared their forthcoming catalog privately with your Snowflake Education account. As the platform admin, acquiring the share is only half the job: you decide **which roles in your account can read it**. You'll grant access, then inspect exactly what that grant did.

We'll cover:
1. Two ways to get data: loading vs. querying in place
2. Public Marketplace vs. Private Listing — and when each fits
3. Searching the public Marketplace — by hand and with CoCo's skills
4. Getting access to a privately shared database
5. Inspecting the shared catalog
6. Your angle — auditing and delegating who can read the share

:books: **By the end you'll be able to:**
- Explain what "zero copy" means and when it's the right choice
- Distinguish Public Marketplace listings from Private Listings
- Search the public Marketplace by hand and with CoCo's `/marketplace-search` and `/get-marketplace-listing-details` skills
- Grant a role access to a privately shared database
- Audit which roles can read a shared database, and delegate least-privilege access to a team via a governed view

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB` and in the `ALPINE_PEAKS_PUBLISHING` listing shared privately with this account.*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> role: ' + my_role + ' | wh: ' + my_wh)

## Set Up Your Workspace
Set your **personal context** — your own role and warehouse — and point reads at the shared course data.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Two Ways to Get Data

Previously you queried data already in Snowflake. Today's question is *where data comes from in the first place*. It arrives in three shapes:

| Shape | Looks like | Roastery & Co. example |
|---|---|---|
| **Structured** | Rows + columns, fixed schema | `ORDERS`, `STORES` |
| **Semi-structured** | Flexible schema (JSON, Parquet, XML) — stored in a [`VARIANT`](https://docs.snowflake.com/en/user-guide/semistructured-intro) column | Loyalty app events |
| **Unstructured** | No schema at all | Supplier invoice PDFs, product photos |

Before worrying about shape, there's a more fundamental question: **do you need to bring it in at all?**

| | Load it in | Query in place (Marketplace / Sharing) |
|---|---|---|
| **You manage** | A pipeline + storage | Nothing — the provider maintains it |
| **Freshness** | As fresh as your last load | Always provider-current |
| **Best for** | Your own source files | Third-party / partner / reference data |

A publisher's book catalog is a good "query in place" case: the publisher maintains it, you just need to read it. The [**Snowflake Marketplace**](https://docs.snowflake.com/en/user-guide/data-marketplace-intro) and **Private Listings** let you access that data as a live, **zero-copy** share — it appears as a database in your account, but no data is copied and you pay no storage for it.

> :bulb: **The mindset shift:** loading (Lab 7) is one option, not the only one. For data a partner already maintains — a publisher's catalog, a supplier's price list, a vendor's reference feed — mounting a share is far less work than building and babysitting a pipeline.

> 🛡️ **Why this matters for admins:** Zero-copy sharing changes your governance surface. You're not managing ETL credentials, staging areas, or copy schedules — but you *are* managing who can read shared data, which roles get access, and whether consumers are querying it responsibly. The attack surface shrinks, but the access-control decisions don't disappear.

## 2 · Public Marketplace vs. Private Listing

Snowflake offers two flavors of zero-copy data sharing:

| | Public Marketplace | Private Listing |
|---|---|---|
| **Who can see it** | Anyone on Snowflake | A curated list of invited accounts |
| **How to find it** | Marketplace (top-level) | Data sharing → External sharing → Shared with you |
| **Typical use** | Weather, demographics, financial data | Partner catalogs, internal divisions, supplier feeds |
| **Access** | Click **Get** in the UI | Provider shares directly; you grant access via SQL |

Today's listing — **Alpine Peaks Publishing** — is a **Private Listing**. Alpine Peaks Publishing (a fictional publisher) has shared their forthcoming catalog privately with the Snowflake Education account. You won't find it on the public Marketplace; it lives in your account's **Shared With You** tab.

:point_right: **Navigate to the listing in a new browser tab** (keep this notebook open):
1. In the Snowsight left navigation, **right-click → Open in new tab** on **Data sharing → External sharing** (so you keep this notebook open).
2. Click the **Shared With You** tab.
3. Find and click the **Alpine Peaks Publishing** listing and review its description.

> 💡 **Already provisioned — just browse.** You'll likely see a **"Get Access"** button — that's expected. In a real scenario, clicking it sends a request to your account administrator to grant you access to this shared database. In this lab, Section 4 below runs a stored procedure that acts as the administrator on your behalf and grants your role the necessary privileges. Once that's done, the button changes to **Open**. For now, just review the listing description and note what's included.

> 📚 [Snowflake Marketplace overview](https://docs.snowflake.com/en/user-guide/data-marketplace) · [Private data sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)

---

![Alpine Peaks Publishing listing](images/ob_pde_alpine_peaks_listing_1_v2.png)

## 3 · Discover Data on the Public Marketplace

You've seen the *concept* of public vs. private sharing. Now see how you'd actually **find** data on the public **Snowflake Marketplace** — the open catalog any Snowflake account can browse.

**Search it yourself in Snowsight** (open a new browser tab, keep this notebook open):
1. In the left navigation, **right-click → Open in new tab** on **Snowflake Marketplace** (you'll see three options: Snowflake Marketplace, Consumer Hub, and Provider Studio — choose **Snowflake Marketplace**).
2. Use the **search bar** at the top to describe what you need — e.g. `coffee`, `foot traffic`, `consumer spending`.
3. Narrow results with the **Provider**, **Category**, and **Business Need** filters on the left.
4. Click any listing to read its description and sample data. For a free listing, **Get** adds it to your account as a zero-copy share.

> :bulb: As the admin, you're often the one who *acquires* listings for the account — weigh cost/terms, decide which roles should get access, and whether to **Get** it at all. Public listings shine for broad **reference data** — weather, demographics, points of interest. Docs: [Exploring listings on the Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace-intro).

### Let CoCo drive the search. 🤖
You don't have to click through the UI — CoCo ships two Marketplace skills. Before using them, ask CoCo to explain how they differ (this is a plain question, so **don't** prefix it with a slash — a slash would launch a skill instead of answering):

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I see two Marketplace skills: /marketplace-search and /get-marketplace-listing-details. How do they differ, and when should I use each one?</pre>

> 🔎 In short: **`/marketplace-search`** casts a wide net and returns a *list* of matching listings; **`/get-marketplace-listing-details`** zooms in on *one* listing you name and explains what's in it and whether it fits your need.

### Search the Marketplace with CoCo. 🤖
Now run a real search for something Roastery-adjacent:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/marketplace-search Are there any coffee-related listings available?</pre>

CoCo returns a short list of coffee- and café-adjacent listings, each with a title, a one-line description, and a link. Skim them and notice how varied third-party data can be.

### Zoom in on one listing. 🤖
Pick a listing from those results and get the full picture with the second skill:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/get-marketplace-listing-details All Coffee Shop Locations in the Top 5 US Cities</pre>

CoCo returns the listing's provider, what data it contains, and how it might (or might not) fit Roastery & Co.'s needs.

> :point_right: **So why aren't we using a public listing today?** Public listings are perfect for broad reference data — but Roastery needs one specific thing: **Alpine Peaks Publishing's forthcoming catalog**, shared *directly and privately* with this account (the listing you reviewed earlier). You won't surface it by searching the public Marketplace — it's private to us. It's the closest fit for the job, so let's grant access and then govern who can read it.

## 4 · Get Access to the Shared Database

The `ALPINE_PEAKS_PUBLISHING` database already exists in this account — the instructor mounted it once. What each learner needs is **read access** to it: a non-owner role can't query a shared database until it's granted privileges on the objects inside.

A stored procedure owned by a privileged role handles this in one call, so you don't need `ACCOUNTADMIN`. You'll see *exactly* what it granted in Section 6:

:point_right: Run the cell below to grant your `_RL` role access to the shared database.

In [ ]:
-- Grant your role access to the shared database (one-time, per user)
CALL PLATPLUS_DB.RESOURCES.ACCESS_ALPINE_PEAKS_PUBLISHING_DATA('{{my_role}}');

> 🔎 **What just happened.** The procedure granted your role **`IMPORTED PRIVILEGES`** on the shared database — the standard consumer-side way to open a database created from a share. When you inspect your grants you'll see them surface as targeted rows — `USAGE` on the database, `USAGE` on the `BOOKS` schema, `SELECT` on the `FORTHCOMING_RELEASES` table — because that's exactly what the **provider** chose to put in the share. You can read that one table, nothing more. (You'll audit these grants yourself in Section 6.)

> :bulb: **How you open a share — and where governance lives.** A consumer can't grant object-level privileges on a shared database — Snowflake blocks it. The one lever is `GRANT IMPORTED PRIVILEGES ON DATABASE … TO ROLE`, which passes through exactly what the **provider** put in the share (here, one table). To go finer: the **provider** can define **database roles** to segment the share (Alpine Peaks hasn't), or you expose a **governed view** in your own account and grant on that — which you'll do in Section 6. The provider controls *what's* in the share; you control *who inside your account* reaches it.

> 🛡️ **How this works — owner's rights delegation.** You don't have permission to grant yourself access to a shared database — that requires ACCOUNTADMIN (or the role that mounted the share). So how did it work?
>
> The stored procedure `ACCESS_ALPINE_PEAKS_PUBLISHING_DATA` runs with **owner's rights** — it executes as the role that owns it (ACCOUNTADMIN), not as your calling role. Internally it runs a single grant:
>
> `GRANT IMPORTED PRIVILEGES ON DATABASE ALPINE_PEAKS_PUBLISHING TO ROLE {{my_role}};`
>
> That's the only consumer-side way to open a shared database. `SHOW GRANTS` then surfaces it as the individual object privileges the provider shared (you'll see this in Section 6).
>
> This is the same pattern you saw in Lab 4 with resource monitors: a privileged procedure acts on behalf of a less-privileged caller. As the admin, this is your design tool — you build the procedure once, and users self-serve safely without needing elevated roles. [Owner's rights procedures](https://docs.snowflake.com/en/developer-guide/stored-procedure/stored-procedures-rights)

## 5 · Inspect the Shared Database

A shared database behaves like any other: it has schemas and tables you can explore with `SHOW` and `SELECT`. Run the three cells below to get oriented before you query.

In [ ]:
-- What schemas are in the shared database?
SHOW SCHEMAS IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
-- What tables are available?
SHOW TABLES IN DATABASE ALPINE_PEAKS_PUBLISHING;

In [ ]:
-- Preview the forthcoming releases catalog
SELECT *
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
LIMIT 10;

> 🔎 **Note the columns.** The table has `TITLE`, `AUTHOR`, `CATEGORY`, and `RELEASE_DATE` — a clean structured catalog of upcoming book releases. No joins needed to start exploring.

## 6 · Audit and Delegate Access

You gave your own role access in Section 4. But as the admin your real job is the *access posture* of the whole account: **who can read this share, and how do you give a team exactly what they need — no more?** Orient in the data first, then audit and delegate.

In [ ]:
-- How many titles are in the catalog?
SELECT COUNT(*)
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES;

In [ ]:
-- What genres does this publisher cover?
SELECT DISTINCT CATEGORY
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
ORDER BY 1;

In [ ]:
-- Which genres have the most forthcoming titles?
SELECT CATEGORY, COUNT(*) AS title_count
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
GROUP BY CATEGORY
ORDER BY title_count DESC;

### Read the access you were actually given. 🤖
Before extending access, know precisely what you hold. Ask CoCo to frame the control model, then inspect your own grants:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I was granted USAGE on a shared database ALPINE_PEAKS_PUBLISHING plus SELECT on one table inside it. As the consumer admin, what do I control vs. the provider, and how would I give a specific team role read access to just that one table — without exposing the whole share?</pre>

> :point_right: Run the two cells below to see exactly what the proc granted your role.

In [ ]:
-- What privileges does your role hold? (includes the grant from Section 4)
SHOW GRANTS TO ROLE {{my_role}};

In [ ]:
-- Filter to every privilege you hold that touches the shared database
SELECT "privilege", "granted_on", "name"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "name" LIKE 'ALPINE_PEAKS_PUBLISHING%'
ORDER BY "granted_on";

> 🔎 **Read your grants.** You'll see **three** rows: `USAGE` on the `DATABASE`, `USAGE` on the `BOOKS` `SCHEMA`, and `SELECT` on the `FORTHCOMING_RELEASES` `TABLE`. That's your single `GRANT IMPORTED PRIVILEGES` **surfacing** the exact objects the provider put in the share — not three grants you made (a consumer can't grant object privileges on shared data). You can read that one table, nothing else in the share.

### Audit who else can reach the share. 🔎
`SHOW GRANTS TO ROLE` answers "what can *I* reach?" As the admin you also need "who in the account can reach *this share*?" That's `SHOW GRANTS ON DATABASE`:

In [ ]:
-- Every role that has been granted access to the shared database
SHOW GRANTS ON DATABASE ALPINE_PEAKS_PUBLISHING;

> 🔎 **Read the posture.** Each row is a role with a foothold into the share (plus one `OWNERSHIP` row — the local `ACCOUNTADMIN` that mounted it). This is the account-wide view an admin uses to answer audits like *"who can see the publisher's data?"* — in one command, no guessing.

### Delegate least-privilege access to a team. 🤖
The café analyst team needs the **Cooking** titles — but they shouldn't get the whole share. The governed pattern: expose a **view** of just what they need, then grant the team role access to the *view only*. Because a view runs with its owner's rights, the team can read your view **without any grant on the share itself**.

The cell below has the `CREATE OR REPLACE VIEW … AS` line pre-filled. Ask CoCo for the **SELECT body**:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">@ALPINE_PEAKS_PUBLISHING Don't run it — write just the SELECT statement (no CREATE, no trailing semicolon) that returns TITLE, AUTHOR, RELEASE_DATE for all Cooking titles from ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES ordered by release_date. I'll paste it as the body of a view my analyst team will read.</pre>

> :point_right: Paste CoCo's SELECT **after** the CREATE line, then run the cell. Reference solution is just below.

In [ ]:
CREATE OR REPLACE VIEW {{my_db}}.PUBLIC.CAFE_TEAM_CATALOG AS
-- Paste CoCo's SELECT statement on the blank line below, then run this cell


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE VIEW {{my_db}}.PUBLIC.CAFE_TEAM_CATALOG AS
    SELECT TITLE, AUTHOR, RELEASE_DATE
    FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
    WHERE CATEGORY = 'Cooking'
    ORDER BY RELEASE_DATE;

</details>

Now grant the analyst team the **least privilege** it needs to use that view: a warehouse to run on, `USAGE` down the path to the view, and `SELECT` on the view itself. Note what's **not** here — no grant on the shared database.

In [ ]:
-- Give the analyst team exactly what it needs: a warehouse + the one view (not the share)
GRANT USAGE ON WAREHOUSE {{my_wh}} TO ROLE {{my_analyst_role}};
GRANT USAGE ON DATABASE {{my_db}} TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.PUBLIC TO ROLE {{my_analyst_role}};
GRANT SELECT ON VIEW {{my_db}}.PUBLIC.CAFE_TEAM_CATALOG TO ROLE {{my_analyst_role}};

### Prove it worked — as the team role. 🔬
Your user also holds your paired analyst role, so you can *become* the team for a moment and confirm the guardrail: it can read your view, but **not** the raw share. Run this — it switches roles, checks both, and switches back for you.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
me = session.get_current_user().strip('"')
analyst, mydb, mywh = me + '_ANALYST_RL', me + '_DB', me + '_WH'
try:
    session.sql(f'USE ROLE {analyst}').collect()
    session.sql(f'USE WAREHOUSE {mywh}').collect()
    n = session.sql(f'SELECT COUNT(*) FROM {mydb}.PUBLIC.CAFE_TEAM_CATALOG').collect()[0][0]
    print(f'✅ As {analyst}: read your governed view — {n} rows')
    try:
        session.sql('SELECT COUNT(*) FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES').collect()
        print(f'⚠️ Unexpected: {analyst} could read the raw share')
    except Exception:
        print(f'✅ As {analyst}: the raw share is correctly BLOCKED — least privilege works')
finally:
    session.sql(f'USE ROLE {me}_RL').collect()
    session.sql(f'USE WAREHOUSE {mywh}').collect()
    print(f'↩️  Role reset to {me}_RL')

### Close the loop — revoke. 🔁
Access is a lifecycle, not a one-way door. When the team no longer needs the view, you revoke it. Run the cell, and the analyst role loses access immediately (the view itself stays, owned by you).

In [ ]:
-- Revoke the team's access to the view (the view remains; only the grant is removed)
REVOKE SELECT ON VIEW {{my_db}}.PUBLIC.CAFE_TEAM_CATALOG FROM ROLE {{my_analyst_role}};

### Trust, but verify — confirm the revoke. 🔬
A revoke isn't done until you've proven it. Become the analyst role once more and try to read the view — this time it should be **blocked**. The cell switches roles, checks, and switches back for you.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
me = session.get_current_user().strip('"')
analyst, mydb, mywh = me + '_ANALYST_RL', me + '_DB', me + '_WH'
try:
    session.sql(f'USE ROLE {analyst}').collect()
    session.sql(f'USE WAREHOUSE {mywh}').collect()
    try:
        session.sql(f'SELECT COUNT(*) FROM {mydb}.PUBLIC.CAFE_TEAM_CATALOG').collect()
        print(f'⚠️ Unexpected: {analyst} can still read the view — re-run the REVOKE cell above, then this one')
    except Exception:
        print(f'✅ Revoke confirmed: {analyst} can no longer read CAFE_TEAM_CATALOG — access is closed')
finally:
    session.sql(f'USE ROLE {me}_RL').collect()
    session.sql(f'USE WAREHOUSE {mywh}').collect()
    print(f'↩️  Role reset to {me}_RL')

> 🔎 **What you just did.** You audited the share account-wide, then delegated a narrow, governed slice to a team — a view they can read *without* any access to the raw share — and revoked it just as cleanly. That grant/revoke lever, over a curated view, is how an admin extends shared data safely.

> 🔎 **Why revoke only the view — not the schema or database?** Revoking `SELECT` on the view is what actually closes off the data. The analyst role still holds `USAGE` on your database and `PUBLIC` schema — but `USAGE` alone exposes *nothing*: a role can't list or query any object it has no object-level grant on. Leaving that harmless plumbing in place lets the team keep using anything else you've deliberately shared. Least privilege is enforced at the `SELECT` grant, and that's gone.

> :bulb: **Provider vs. consumer control.** The **provider** (Alpine Peaks) decides *what* is in the share and can revoke it at any time. As the **consumer admin**, you decide *who inside your account* reads it — and to narrow *exactly what* they see, you expose a **governed view** over the share and grant on that (a consumer can't grant object-level privileges on shared data directly). Provider controls the data; you control internal access. That split is the governance model for all shared data.

## Check Your Work
Run the next cell to confirm your role can reach the shared database. Prints a ✅ / ⛔ for each check — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
# The delegation proof cell switches roles — make sure we evaluate as your own role
session.sql(f'USE ROLE {user}_RL').collect()
session.sql(f'USE WAREHOUSE {user}_WH').collect()
checks = [
    ("You can reach the shared Roastery data",
     "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.STORES", 1),
    ("Your role can query the Alpine Peaks Publishing catalog",
     "SELECT COUNT(*) FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES", 1),
    ("Your governed CAFE_TEAM_CATALOG view returns rows",
     f"SELECT COUNT(*) FROM {my_db}.PUBLIC.CAFE_TEAM_CATALOG", 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not accessible — re-run the steps above)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Check your understanding. 🧠
A few quick questions on this lab's key themes. Run each cell below, choose an answer, and the feedback appears inline — no grade, just a self-check.

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- **Zero copy** means no data moves — a shared database is a live window onto the provider's data, no storage cost on your side
- The **public Marketplace** is self-serve — search it in Snowsight, or drive it with CoCo's `/marketplace-search` (find listings) and `/get-marketplace-listing-details` (inspect one)
- **Private Listings** are for curated, account-specific data sharing — not visible on the public Marketplace
- **Access came via `IMPORTED PRIVILEGES`:** the only consumer-side way to open a share — it surfaces exactly what the provider put in it (one table). A consumer can't grant object-level privileges on shared data directly
- **Audit with `SHOW GRANTS`:** `SHOW GRANTS TO ROLE` shows what a role holds; `SHOW GRANTS ON DATABASE` shows every role that can reach a shared database
- **Delegate least-privilege via a view:** expose a governed view and grant a team `SELECT` on it — they read your curated slice, never the raw share (owner's rights) — and `REVOKE` closes the loop
- **Provider vs. consumer control:** the provider decides what's in the share; you decide who inside your account can read it

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.